# Independent verification of the Table 3 correction

This notebook deliberately reuses **none** of the analysis code. It needs only `numpy`, `pandas` and
`scipy`, all preinstalled in Colab — nothing to install, nothing to restart.

**The claim under test.** The August version of `downstream_transfer.py` loaded
`joint_language_table.csv`, whose `retrieval` column is a single number per language: the mean
pooled across all four encoders, and correlated each encoder's transfer scores against that same
pooled vector. If so, the August Table 3 controlled for the wrong thing, and multilingual-E5's
partial correlation is about −0.35 rather than −0.02.

**Section 2 settles it on its own** and uses no partial correlations at all — just
`scipy.stats.spearmanr`. Sections 3–5 quantify the consequence.

August Table 3:

| Encoder | ρ_ret | ρ_fert | ρ_fert\|ret |
|---|---|---|---|
| LaBSE | +0.22 | −0.10 | −0.04 |
| BGE-M3 | +0.76 | −0.38 | +0.04 |
| MPNet | +0.78 | −0.49 | −0.20 |
| mult.-E5 | +0.68 | −0.37 | −0.02 |

Run the cells in order.

## 0. Load the four files

Run the next cell and pick all four at once in the file dialog:

- `downstream_all_encoders.csv`
- `retrieval_per_language.csv`
- `joint_language_table.csv`
- `fertility_production_v3_spm.csv`

If you would rather not use the upload dialog, skip this cell and set `DATA` in the cell after it to
a folder that contains them.

In [ ]:
try:
    from google.colab import files
    files.upload()
except Exception as e:
    print('Upload dialog unavailable, that is fine:', e)
    print('Set DATA in the next cell to a folder containing the four CSVs.')

In [ ]:
import os
import numpy as np, pandas as pd
from scipy import stats

DATA = '.'          # <-- change this if your files are elsewhere

NEEDED = ['downstream_all_encoders.csv', 'retrieval_per_language.csv',
          'joint_language_table.csv', 'fertility_production_v3_spm.csv']

def find(fname):
    for root, _, fs in os.walk(DATA):
        if fname in fs:
            return os.path.join(root, fname)
    raise FileNotFoundError(
        f"Could not find {fname} under {os.path.abspath(DATA)}.\n"
        f"Files I can see here: {sorted(os.listdir(DATA))[:20]}")

paths = {}
missing = []
for f in NEEDED:
    try:
        paths[f] = find(f)
    except FileNotFoundError as e:
        missing.append(f); print('MISSING:', f)
if missing:
    raise SystemExit('Upload the missing files, then re-run this cell.')

down  = pd.read_csv(paths['downstream_all_encoders.csv'])
retr  = pd.read_csv(paths['retrieval_per_language.csv'])
joint = pd.read_csv(paths['joint_language_table.csv'])
fert  = pd.read_csv(paths['fertility_production_v3_spm.csv'])

for nm, df in [('downstream', down), ('retrieval', retr), ('joint', joint), ('fertility', fert)]:
    print(f'{nm:11s} {df.shape[0]:5d} rows   columns: {list(df.columns)[:6]}')

## 1. Look at the file before computing anything

How many retrieval columns does `joint_language_table.csv` have? If exactly one, a per-encoder table
built from it cannot have used each encoder's own retrieval, because those four numbers are not in
the file.

In [ ]:
print('columns:', list(joint.columns))
print('retrieval-like columns:', [x for x in joint.columns if 'retriev' in x.lower()])

ENC = {'sentence-transformers/LaBSE': 'labse',
       'BAAI/bge-m3': 'bgem3',
       'intfloat/multilingual-e5-base': 'e5',
       'sentence-transformers/paraphrase-multilingual-mpnet-base-v2': 'mpnet'}

unmapped = set(retr.model.unique()) - set(ENC)
if unmapped:
    print('\nWARNING: unmapped model names in retrieval_per_language.csv:', unmapped)

wide = (retr.assign(e=retr.model.map(ENC))
            .dropna(subset=['e'])
            .pivot(index='language', columns='e', values='mean_top1'))

chk = joint.set_index('language')[['retrieval']].join(wide)
chk['pooled_mean'] = chk[['labse', 'bgem3', 'e5', 'mpnet']].mean(axis=1)
diff = (chk.retrieval - chk.pooled_mean).abs()
print(f'\nmax |retrieval - mean(4 encoders)| = {diff.max():.12f}   (n={diff.notna().sum()})')
print(f'identical for every language: {bool((diff < 1e-12).all())}')
print()
print(chk.head(5).round(6).to_string())

## 2. The decisive test — plain Spearman, no partial correlations

The ρ_ret column of Table 3 is an ordinary Spearman correlation. "Retrieval" can mean only one of
two things. Compute both, compare each to what you published, and whichever matches is what the
paper did.

In [ ]:
PAPER = {'labse': {'ret': 0.22, 'fert': -0.10, 'part': -0.04},
         'bgem3': {'ret': 0.76, 'fert': -0.38, 'part': +0.04},
         'mpnet': {'ret': 0.78, 'fert': -0.49, 'part': -0.20},
         'e5':    {'ret': 0.68, 'fert': -0.37, 'part': -0.02}}
NAME = {'labse': 'LaBSE', 'bgem3': 'BGE-M3', 'mpnet': 'MPNet', 'e5': 'mult.-E5'}

J = joint.merge(wide, left_on='language', right_index=True)

rows, err_pool, err_own = [], 0.0, 0.0
for enc in ['labse', 'bgem3', 'mpnet', 'e5']:
    d = down[down.encoder == enc][['language', 'macro_f1']]
    s = J.merge(d, on='language')
    r_pool = stats.spearmanr(s.macro_f1, s.retrieval)[0]
    r_own  = stats.spearmanr(s.macro_f1, s[enc])[0]
    err_pool += abs(r_pool - PAPER[enc]['ret'])
    err_own  += abs(r_own  - PAPER[enc]['ret'])
    rows.append({'encoder': NAME[enc], 'n': len(s), 'paper': PAPER[enc]['ret'],
                 'pooled control': round(r_pool, 3), 'own-encoder control': round(r_own, 3)})

print(pd.DataFrame(rows).to_string(index=False))
print('\nTotal absolute error against the published column:')
print(f'   pooled control      : {err_pool:.3f}')
print(f'   own-encoder control : {err_own:.3f}')
print('\nThe smaller number identifies what the paper actually computed.')

**If the pooled column matches and the own-encoder column does not**, the diagnosis is confirmed and
the corrected Table 3 in the paper stands.

**If the own-encoder column matched instead**, the August table would have been right and the
correction wrong.

## 3. Two independent partial-correlation implementations

Before trusting either, check both on synthetic data with a known answer. The two methods are
genuinely different: one combines three pairwise rank correlations, the other regresses ranks and
correlates the residuals.

In [ ]:
def partial_formula(x, y, z):
    "Spearman partial correlation from three pairwise rank correlations."
    rxy = stats.spearmanr(x, y)[0]
    rxz = stats.spearmanr(x, z)[0]
    ryz = stats.spearmanr(y, z)[0]
    return (rxy - rxz * ryz) / np.sqrt((1 - rxz**2) * (1 - ryz**2))

def partial_residual(x, y, z):
    "Same quantity via regression of ranks on ranks, then Pearson on residuals."
    rx, ry, rz = (stats.rankdata(v) for v in (x, y, z))
    Z = np.column_stack([np.ones(len(rz)), rz])
    def resid(a):
        beta, *_ = np.linalg.lstsq(Z, a, rcond=None)
        return a - Z @ beta
    return stats.pearsonr(resid(rx), resid(ry))[0]

rng = np.random.default_rng(0)
z = rng.normal(size=400)
x = z + rng.normal(scale=.5, size=400)
y_null = z + rng.normal(scale=.5, size=400)           # depends on z only
y_real = z + 0.8 * x + rng.normal(scale=.5, size=400) # also depends on x

for label, y in [('should be about 0', y_null), ('should be clearly positive', y_real)]:
    a, b = partial_formula(x, y, z), partial_residual(x, y, z)
    print(f'{label:28s} formula {a:+.3f}   residual {b:+.3f}   agree={abs(a-b) < 0.01}')
print('\nBoth must agree and behave as labelled before the next cell means anything.')

## 4. The partial correlations, both controls, both implementations

In [ ]:
rows = []
for enc in ['labse', 'bgem3', 'mpnet', 'e5']:
    d = down[down.encoder == enc][['language', 'macro_f1']]
    s = (J.merge(d, on='language')
          .merge(fert[['language', 'gpt4o_o200k_tok_per_verse']], on='language'))
    t = s.macro_f1.values
    f = s.gpt4o_o200k_tok_per_verse.values
    rows.append({'encoder': NAME[enc], 'n': len(s), 'paper': PAPER[enc]['part'],
                 'pooled (formula)':  round(partial_formula(t, f, s.retrieval.values), 3),
                 'pooled (residual)': round(partial_residual(t, f, s.retrieval.values), 3),
                 'own (formula)':     round(partial_formula(t, f, s[enc].values), 3),
                 'own (residual)':    round(partial_residual(t, f, s[enc].values), 3)})

print(pd.DataFrame(rows).to_string(index=False))
print('\nThe two implementations must match within rounding in every column.')
print('Then compare the "paper" column against "pooled" and against "own".')

## 5. Bootstrap interval for the corrected values

In [ ]:
rng = np.random.default_rng(42)
rows = []
for enc in ['labse', 'bgem3', 'mpnet', 'e5']:
    d = down[down.encoder == enc][['language', 'macro_f1']]
    s = (J.merge(d, on='language')
          .merge(fert[['language', 'gpt4o_o200k_tok_per_verse']], on='language'))
    t, f, r = s.macro_f1.values, s.gpt4o_o200k_tok_per_verse.values, s[enc].values
    point = partial_formula(t, f, r)
    n = len(s)
    boot = []
    for _ in range(10000):
        i = rng.integers(0, n, n)
        try:
            boot.append(partial_formula(t[i], f[i], r[i]))
        except Exception:
            pass
    lo, hi = np.percentile(boot, [2.5, 97.5])
    rows.append({'encoder': NAME[enc], 'n': n, 'partial': round(point, 3),
                 'ci_lo': round(lo, 3), 'ci_hi': round(hi, 3),
                 'excludes_zero': bool(lo > 0 or hi < 0)})

print(pd.DataFrame(rows).to_string(index=False))
nn = 40
print(f'\nSmallest |partial rho| detectable at n={nn}, alpha=0.05: '
      f'{np.tanh(stats.norm.ppf(0.975) / np.sqrt(nn - 4)):.3f}')
print('Compare that with the interval widths: the near-zero values bound the effect, not exclude it.')

## 6. Where the pooled control came from

Everything above is statistical. The direct evidence is in `04_flores_replication_and_downstream.ipynb`:
the per-encoder correlations there use the `retrieval` column of `joint_language_table.csv`, a
single vector (the mean over all four encoders) reused for every encoder row. The corrected
Table 3 controls for each encoder's own retrieval accuracy; `code/table3_partial_correlations.py`
computes it from the shipped results in one command.
